In [0]:
# Create widgets for parameters
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "lab_data")

# set widgets
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# read the table
df_silver = spark.read.table(f"{catalog}.{schema}.movies_silver")

In [0]:
from pyspark.sql.functions import col, hash, monotonically_increasing_id, split, trim, upper

display(df_silver.limit(5))

In [0]:
%sql
-- Create or replace dimension table using dynamic catalog and schema parameters
CREATE OR REPLACE TABLE ${catalog}.${schema}.dim_movies AS 
SELECT 
    md5(concat(coalesce(title, ''), coalesce(country, ''))) AS movie_id,
    title,
    country,
    director,
    cast
FROM (
  SELECT DISTINCT title, country, director, cast
  FROM ${catalog}.${schema}.movies_silver
  WHERE title IS NOT NULL
)
QUALIFY ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY title) = 1;

In [0]:
%sql
Select * from ${catalog}.${schema}.dim_movies
limit 10;

In [0]:
%sql

CREATE OR REPLACE TABLE ${catalog}.${schema}.dim_genres AS --- creating the 2 table for genres
SELECT 
    md5(trim(genre_name)) AS genre_id,
    trim(genre_name) AS genre_name --- space remove with trim to do not create duplicates in genres
FROM (
    SELECT DISTINCT explode(split(genres, ',')) AS genre_name
    FROM ${catalog}.${schema}.movies_silver
    WHERE genres IS NOT NULL
);

In [0]:
%sql
Select * from ${catalog}.${schema}.dim_genres
limit 10;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.${schema}.fact_movie_performance AS --- creating table to check the data realted to finances of movies
SELECT 
    md5(concat(coalesce(s.title, ''), coalesce(s.country, ''))) AS movie_id,
    s.title,
    s.genres,
    s.country,
    s.director,
    s.cast,
    s.release_year,
    s.rating_out_of_10 AS rating,
    COALESCE(s.budget_usd, 0.0) AS budget,
    COALESCE(s.box_office_usd, 0.0) AS revenue,
    (COALESCE(s.box_office_usd, 0.0) - COALESCE(s.budget_usd, 0.0)) AS profit,
    current_timestamp() AS created_at
FROM ${catalog}.${schema}.movies_silver s;

In [0]:
%sql
Select * from ${catalog}.${schema}.fact_movie_performance
limit 10;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.${schema}.gold_genre_revenue_summary AS
WITH exploded_genres AS (
    SELECT 
        f.movie_id,
        f.revenue,
        f.budget,
        f.profit,
        f.rating,
        trim(genre_item) AS genre_name
    FROM ${catalog}.${schema}.fact_movie_performance f
    LATERAL VIEW explode(split(f.genres, ',')) AS genre_item
)
SELECT 
    g.genre_name,
    COUNT(DISTINCT e.movie_id) AS total_movies,
    SUM(e.revenue) AS total_revenue,
    SUM(e.budget) AS total_budget,
    AVG(e.profit) AS avg_profit,
    ROUND(AVG(e.rating), 2) AS avg_rating
FROM exploded_genres e
INNER JOIN ${catalog}.${schema}.dim_genres g ON e.genre_name = g.genre_name
GROUP BY g.genre_name;

In [0]:
%sql
Select * from ${catalog}.${schema}.gold_genre_revenue_summary
limit 10;

In [0]:
%sql
-- 1. Column-Level Security (CLS) / Masking:
CREATE OR REPLACE FUNCTION ${catalog}.${schema}.revenue_mask(revenue DOUBLE)
RETURN CASE 
    WHEN is_account_group_member('admin') THEN revenue -- if you are admin you can see revenue, otherwise only 0
    ELSE 0.0 
END;

ALTER TABLE ${catalog}.${schema}.fact_movie_performance 
ALTER COLUMN revenue SET MASK ${catalog}.${schema}.revenue_mask;

-- 2. Row-Level Security (RLS):
CREATE OR REPLACE FUNCTION ${catalog}.${schema}.us_movies_filter(country STRING)
RETURN is_account_group_member('admin') OR country = 'United States'; -- if you are admin you can see all movies, otherwise only US movies

ALTER TABLE ${catalog}.${schema}.fact_movie_performance 
SET ROW FILTER ${catalog}.${schema}.us_movies_filter ON (country);

# Check if Masking working in the revenue context:

In [0]:
%sql
SELECT title, country, revenue, profit  -- seems working :)
FROM ${catalog}.${schema}.fact_movie_performance
LIMIT 10;